# Forecast workflow on simulated series

Dr. Pavanam Thomas  
Time-Series Forecasting Lab  
Copyright 2026. MIT License.

This notebook follows the protocol in `FORECAST_VALIDATION_PLAYBOOK.md`. Every series is **simulated**. In-sample fitted values are not out-of-sample forecasts. A named model is compared to a naive benchmark on the same origin and horizon.

Related repositories: [quantitative-finance-models](https://github.com/pavanamthomas/quantitative-finance-models), [statistical-reasoning-validation](https://github.com/pavanamthomas/statistical-reasoning-validation).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from tsforecast.arima_models import ARIMAForecaster, fit_arima, residual_diagnostics
from tsforecast.dgp import DEFAULT_SEED, generate_catalog
from tsforecast.eda import seasonal_profile, series_summary, transform_report
from tsforecast.metrics import error_summary
from tsforecast.smoothing import HoltWintersForecaster
from tsforecast.stationarity import acf_pacf_table, adf_unit_root
from tsforecast.validation import (
    LinearTrendForecaster,
    MeanForecaster,
    NaiveForecaster,
    SeasonalNaiveForecaster,
    chow_test,
    evaluate_forecaster,
    forecast_from_origin,
)
from tsforecast.volatility import fit_garch11, squared_acf_lag1

print("Laboratory seed:", DEFAULT_SEED)
catalog = generate_catalog(seed=DEFAULT_SEED)
list(catalog)

## I inspect the series

The five DGPs are labelled `simulated_*`. Inspection here is descriptive. It does not establish forecast skill.

In [ ]:
summaries = pd.DataFrame({name: series_summary(item.values) for name, item in catalog.items()}).T
summaries

In [ ]:
fig, axes = plt.subplots(len(catalog), 1, figsize=(9, 10))
for ax, (name, item) in zip(axes, catalog.items()):
    ax.plot(item.values.index, item.values.to_numpy(), color="#1f4e79", linewidth=1.0)
    ax.set_title(f"{name} (simulated)")
fig.tight_layout()
plt.show()

for name, item in catalog.items():
    print(name, item.dgp)

## I define the forecast horizon

Hold out the last twelve observations of the simulated seasonal series. The origin is the last training index. Training uses only observations at or before that origin.

In [ ]:
seasonal = catalog["simulated_seasonal"].values
HORIZON = 12
ORIGIN = len(seasonal) - HORIZON - 1
print("origin iloc:", ORIGIN, "label:", seasonal.index[ORIGIN])
print("train ends at", seasonal.index[ORIGIN], "; test is the next", HORIZON, "points")

## I establish a naive benchmark

Seasonal naive is the relevant naive method for a stable monthly pattern. Last-value naive and the expanding mean are reported on the same origin so that later models are not compared to an invisible baseline.

In [ ]:
naive_rows = []
for factory in (NaiveForecaster, MeanForecaster, lambda: SeasonalNaiveForecaster(period=12)):
    fc, train, test = forecast_from_origin(seasonal, factory(), ORIGIN, HORIZON)
    row = {"model": fc.name, **error_summary(test.to_numpy(), fc.point, train.to_numpy(), seasonality=12)}
    naive_rows.append(row)
    print(fc.name, "interval assumption:", fc.interval_note)
pd.DataFrame(naive_rows)

## I identify transformations

ADF is a unit-root diagnostic. The regression specification must match the alternative. ACF/PACF describe linear dependence in the working series.

In [ ]:
print(transform_report(seasonal, period=12))
print(seasonal_profile(seasonal, period=12))
adf = adf_unit_root(seasonal, regression="c")
print(adf)
acf_pacf_table(seasonal, nlags=15).head(13)

On the simulated AR(1), ADF with a constant is expected to reject a unit root in this sample size. That rejection is not a forecast result.

In [ ]:
ar = catalog["simulated_stationary_ar"].values
print(adf_unit_root(ar, regression="c"))

## I estimate candidate models

Fits below use only the training slice ending at `ORIGIN`.

In [ ]:
hw = HoltWintersForecaster(trend=None, seasonal="add", seasonal_periods=12)
fc_hw, train, test = forecast_from_origin(seasonal, hw, ORIGIN, HORIZON)
nonseasonal = ARIMAForecaster(order=(1, 0, 0), name="ARIMA(1,0,0)_nonseasonal")
fc_ar, _, _ = forecast_from_origin(seasonal, nonseasonal, ORIGIN, HORIZON)
sarima = ARIMAForecaster(order=(0, 0, 1), seasonal_order=(1, 0, 0, 12), name="SARIMA_seasonal")
fc_sa, _, _ = forecast_from_origin(seasonal, sarima, ORIGIN, HORIZON)

cand = []
for fc in (fc_hw, fc_ar, fc_sa):
    cand.append({"model": fc.name, **error_summary(test.to_numpy(), fc.point, train.to_numpy(), seasonality=12)})
pd.concat([pd.DataFrame(naive_rows), pd.DataFrame(cand)], ignore_index=True)

## I inspect residuals

Ljung–Box is computed on the estimation-window residuals. It is a specification check, not an out-of-sample score.

In [ ]:
fit_sa = fit_arima(train, order=(0, 0, 1), seasonal_order=(1, 0, 0, 12))
residual_diagnostics(fit_sa, lags=12)

## I perform rolling evaluation

Each origin refits a new model on data at or before that origin. If this step used the full sample, `tests/test_forecast_no_leakage.py` would fail.

In [ ]:
rolling = []
for factory in (
    NaiveForecaster,
    lambda: SeasonalNaiveForecaster(period=12),
    MeanForecaster,
    lambda: HoltWintersForecaster(trend=None, seasonal="add", seasonal_periods=12),
):
    tbl = evaluate_forecaster(
        seasonal,
        factory,
        min_train=72,
        horizon=12,
        step=24,
        seasonality=12,
    )
    rolling.append(tbl)
roll = pd.concat(rolling, ignore_index=True)
roll.groupby("model")[["rmse", "mae", "mase"]].mean()

## I compare errors and I assess stability

A linear trend DGP is the place to see undifferenced ARIMA look plausible in sample. A known mean shift is the place to see a pooled mean mix regimes. Both series are simulated.

In [ ]:
trend = catalog["simulated_trend_only"].values
origin_t = len(trend) - 25
rows = []
for factory in (
    NaiveForecaster,
    MeanForecaster,
    LinearTrendForecaster,
    lambda: ARIMAForecaster(order=(1, 0, 0), name="ARIMA(1,0,0)_no_difference"),
    lambda: ARIMAForecaster(order=(0, 1, 1), name="ARIMA(0,1,1)"),
):
    fc, tr, te = forecast_from_origin(trend, factory(), origin_t, 24)
    rows.append({"model": fc.name, **error_summary(te.to_numpy(), fc.point, tr.to_numpy())})
pd.DataFrame(rows)

In [ ]:
broke = catalog["simulated_structural_break"]
y_b = broke.values
tb = int(broke.parameters["break_index"])
print(chow_test(y_b, tb, trend=False))
fc_pre, tr_pre, te_post = forecast_from_origin(y_b, MeanForecaster(), tb - 1, 24)
error_summary(te_post.to_numpy(), fc_pre.point, tr_pre.to_numpy())

## I report forecast uncertainty

Interval construction is part of the model. ARIMA intervals here are Gaussian analytic intervals from `statsmodels`. Holt–Winters intervals are residual simulations. Neither is distribution-free.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(train.index[-48:], train.iloc[-48:], color="#1f4e79", label="train (simulated)")
ax.plot(test.index, test.to_numpy(), color="black", label="held-out (simulated)")
ax.plot(test.index, fc_hw.point, color="#c45911", label=fc_hw.name)
ax.fill_between(test.index, fc_hw.lower, fc_hw.upper, color="#c45911", alpha=0.2)
ax.set_title("Holt–Winters interval on a simulated seasonal series")
ax.legend()
fig.tight_layout()
plt.show()
print(fc_hw.interval_note)

## Volatility clustering (simulated GARCH(1,1))

Gaussian GARCH(1,1) quasi-likelihood, zero conditional mean, covariance stationarity `alpha + beta < 1`. This is not a trading model and not an `arch`-package wrapper.

In [ ]:
r = catalog["simulated_volatility_clustering"].values
print("squared-return ACF lag 1 (simulated):", squared_acf_lag1(r.to_numpy()))
est = fit_garch11(r.to_numpy())
{
    "omega_hat": est.omega,
    "alpha_hat": est.alpha,
    "beta_hat": est.beta,
    "persistence": est.persistence,
    "success": est.success,
}

## I state limitations

- All paths are simulated. External validity is not claimed.
- Orders are chosen to match or to misspecify a known DGP; there is no automated search.
- ADF non-rejection is not proof of a unit root.
- Chow statistics here condition on a known break date.
- Forecast intervals inherit Gaussian (or residual-bootstrap) assumptions.
- Rolling evaluation in this notebook uses a coarse step for speed. `scripts/run_all.py` writes the laboratory tables used in CI.

Numeric artifacts after a full run live in `outputs/tables/` and `outputs/figures/`. They must be regenerated; they are not observational findings.